In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.append("../src")

from dotenv import load_dotenv
load_dotenv("../.env")

from indexer import get_chroma_collection
from hybrid_retriever import load_chunks_for_bm25, BM25Index, hybrid_search, canonicalize_query
from reranker import rerank_with_fallback
from generator import RAGGenerator

collection = get_chroma_collection(collection_name="essai_custom", force_recreate=False, mode="query")
chunks_list = load_chunks_for_bm25()
chunks_by_id = {c["chunk_id"]: c for c in chunks_list}
bm25_index = BM25Index(chunks_list)

generator = RAGGenerator(max_memory_turns=5)
print("✅ Pipeline complet initialisé")

c:\Users\USER\OneDrive\Documents\ESSAI-assistant\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3816.09it/s]


✅ 4 clé(s) Cerebras chargée(s)
✅ Pipeline complet initialisé


In [2]:
def ask(session_id, question, k_hybrid=10, k_final=5):
    canonical_q = canonicalize_query(question)
    
    hybrid_results, status = hybrid_search(
        collection, bm25_index, chunks_by_id, canonical_q, k=k_hybrid
    )
    
    if status == "hors_corpus":
        answer = ("Je ne trouve pas d'information sur ce sujet dans les documents de l'ESSAI. "
                   "Essayez de reformuler votre question.")
        print(f"Q: {question}\nR: {answer}\n")
        return answer

    final_chunks = rerank_with_fallback(canonical_q, hybrid_results, top_k=k_final)

    answer = generator.generate(session_id, question, final_chunks)

    print(f"Q: {question}")
    print(f"R: {answer}\n")
    return answer

In [15]:

session = "test_etudiant_1"

ask(session, "c'est quand la deadline pour déposer le sujet de pfe")
ask(session, "et pour le rapport de mi parcours ?")
ask(session, "quelle sont les papiers de pfe ")
ask(session, "que faire ASIA ?")
ask(session, "les options de eme ?")
ask(session, "les reglements interieurs?")


⚠️ Échec avec une clé (essai suivant) : 402 Client Error: Payment Required for url: https://api.cerebras.ai/v1/chat/completions
⚠️ Échec avec une clé (essai suivant) : 402 Client Error: Payment Required for url: https://api.cerebras.ai/v1/chat/completions
⚠️ Échec avec une clé (essai suivant) : 402 Client Error: Payment Required for url: https://api.cerebras.ai/v1/chat/completions
Q: c'est quand la deadline pour déposer le sujet de pfe
R: Je ne trouve pas la date limite précise pour le dépôt du sujet de PFE dans les documents fournis. Il est impératif de respecter scrupuleusement les délais indiqués dans le calendrier des PFE communiqué par l'administration via l'affichage à l'école.

⚠️ Échec avec une clé (essai suivant) : 402 Client Error: Payment Required for url: https://api.cerebras.ai/v1/chat/completions
⚠️ Échec avec une clé (essai suivant) : 402 Client Error: Payment Required for url: https://api.cerebras.ai/v1/chat/completions
Q: et pour le rapport de mi parcours ?
R: Le rappo

"Le règlement intérieur stipule qu'il est impératif de préserver les bâtiments, les équipements et le matériel de l'école, toute dégradation exposant son auteur à des sanctions et à l'obligation de réparer les dommages. Il est strictement interdit de fumer dans l'ensemble des espaces de l'établissement, y compris les salles de cours et les laboratoires. Enfin, les étudiants ne peuvent pas demeurer dans les locaux en dehors des horaires réservés aux études sans une autorisation préalable de l'administration."

In [4]:
generator.reset_session(session)
print("Mémoire réinitialisée")

Mémoire réinitialisée


In [17]:

session = "test_etudiant_1"

ask(session, "ca vas")



Q: ca vas
R: Je ne trouve pas cette information dans les documents disponibles concernant l'ESSAI. Je suis un assistant dédié aux questions administratives et académiques de l'ESSAI.



"Je ne trouve pas cette information dans les documents disponibles concernant l'ESSAI. Je suis un assistant dédié aux questions administratives et académiques de l'ESSAI."

In [19]:
from greetings import detect_greeting, get_greeting_response
GREETING_TEST_CASES = [
    # Correct, sans faute
    "bonjour",
    "salut",
    "merci",
    # Avec fautes de frappe (le fuzzy doit rattraper)
    "bonjr",
    "bnjour",
    "slaut",
    "mercii",
    "boujour",
    "salu",
    # Derja / arabizi
    "salem",
    "labes",
    "sava",
    # Multi-mots
    "au revoir",
    "comment vas tu",
    "qui es tu",
    "que peux tu faire",
    # Ne doivent PAS être détectés comme small talk (vraies questions)
    "bonjour, c'est quand les inscriptions ?",
    "salut je voudrais savoir la date limite du pfe",
    "combien de crédits maximum je peux avoir",
    "c'est quoi le pfa",
]

for q in GREETING_TEST_CASES:
    category = detect_greeting(q)
    status = f"→ SMALL TALK ({category})" if category else "→ RAG déclenché"
    print(f"{q!r:<55} {status}")

'bonjour'                                               → SMALL TALK (salutation)
'salut'                                                 → SMALL TALK (salutation)
'merci'                                                 → SMALL TALK (remerciement)
'bonjr'                                                 → SMALL TALK (salutation)
'bnjour'                                                → SMALL TALK (salutation)
'slaut'                                                 → SMALL TALK (salutation)
'mercii'                                                → SMALL TALK (remerciement)
'boujour'                                               → SMALL TALK (salutation)
'salu'                                                  → SMALL TALK (salutation)
'salem'                                                 → SMALL TALK (salutation)
'labes'                                                 → SMALL TALK (comment_ca_va)
'sava'                                                  → SMALL TALK (comment_ca_va)
'au re

In [23]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../src")

from dotenv import load_dotenv
load_dotenv("../.env")

from indexer import get_chroma_collection
from hybrid_retriever import load_chunks_for_bm25, BM25Index, hybrid_search, canonicalize_query
from reranker import rerank_with_fallback
from generator import RAGGenerator
from local_llm import OllamaClient

collection = get_chroma_collection(collection_name="essai_custom", force_recreate=False, mode="query")
chunks_list = load_chunks_for_bm25()
chunks_by_id = {c["chunk_id"]: c for c in chunks_list}
bm25_index = BM25Index(chunks_list)

ollama_check = OllamaClient()
print("Ollama disponible :", ollama_check.is_available())

generator_cerebras = RAGGenerator(provider="cerebras", max_memory_turns=5)
generator_ollama = RAGGenerator(provider="ollama", max_memory_turns=5, ollama_model="qwen2.5:latest")

print("✅ Générateurs initialisés")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4927.13it/s]


Ollama disponible : True
✅ 4 clé(s) Cerebras chargée(s)
✅ Générateurs initialisés
